# 06 · Shuffle, Wide vs. Narrow, Broadcast Join (Case B)

**Theory**: docs/03-transformations-actions-dag.md, docs/06-persistence-and-optimization.md

**Prerequisite**: `make up-cluster` still running.

This lab puts a Shuffle-free join (`empresas`, 50 rows -> `broadcast()`)
side by side with a Shuffle-heavy one (`funcionarios`, thousands of
rows -> SortMergeJoin), so you can see the cost difference in
the Spark UI, not just in theory.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import sum as spark_sum

spark = get_connect_session("06-shuffle-broadcast")

vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
empresas = spark.read.parquet(layer_path("connect", "bronze", "empresas"))
funcionarios = spark.read.parquet(layer_path("connect", "bronze", "funcionarios"))

## Broadcast join — no Shuffle of `vendas`

Look for `BroadcastHashJoin` in the plan below. Then check the Spark UI
(http://localhost:4040) Stages tab for this job: no `Exchange` step for the
large side.

In [ ]:
broadcast_join = vendas.join(broadcast(empresas), "id_empresa")
broadcast_join.explain()
broadcast_join.groupBy("setor").agg(spark_sum("valor")).count()

## Shuffle join — both sides get redistributed

Look for `SortMergeJoin` and (usually two) `Exchange` nodes — one per side of
the join. In the Spark UI, compare this Stage's duration against the
broadcast join above.

In [ ]:
shuffle_join = vendas.join(funcionarios, "id_funcionario")
shuffle_join.explain()
shuffle_join.groupBy("cargo").agg(spark_sum("valor")).count()

## Repartitioning once, reusing across operations

If you know you'll run several `groupBy("id_empresa")` queries in a row,
repartitioning by that key upfront avoids paying the Shuffle cost more than
once.

In [ ]:
vendas_by_empresa = vendas.repartition(8, "id_empresa")
vendas_by_empresa.cache()
vendas_by_empresa.count()  # materialize the cache

# Both of these reuse the same partitioning — no extra shuffle
vendas_by_empresa.groupBy("id_empresa").agg(spark_sum("valor")).show()
vendas_by_empresa.groupBy("id_empresa").count().show()

vendas_by_empresa.unpersist()

In [ ]:
spark.stop()